In [1]:
"""
MAP - Charting Student Math Misunderstandings - Inference v5.2 (BM25 Candidate Expansion)
基於 v5.1 (0.84) 的改進：
  - Prompt 完全不變（避免 v7.0 截斷問題）
  - 新增 BM25 候選擴充：beam search candidates + BM25 取回相似訓練例 labels
  - Phase 2 對所有候選（beam + retrieval）一起評分，提升召回
  - 移除 v7.0 的 normalize_label（會把錯誤輸出強行映射到錯誤 label）
"""

import re
import math
import time
import pandas as pd
import torch
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from tqdm import tqdm

# ==== 路徑 ====
BASE_MODEL_PATH = "/kaggle/input/models/google/gemma-3/transformers/gemma-3-1b-it/1"
ADAPTER_PATH    = "/kaggle/input/datasets/alextsai2004/gemma-math-misunderstanding-lora/best_gemma_lora_model"
TEST_CSV        = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/test.csv"
TRAIN_CSV       = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/train.csv"
SAMPLE_CSV      = "/kaggle/input/competitions/map-charting-student-math-misunderstandings/sample_submission.csv"
OUTPUT_CSV      = "/kaggle/working/submission.csv"

# ==== 超參 ====
BATCH_SIZE     = 16   # Phase 1 每批筆數
SUB_BATCH_SIZE = 32   # Phase 2 forward 子批上限，防 OOM
NUM_BEAMS      = 5
NUM_RETURN     = 5
MAX_NEW_TOKENS = 24
BM25_TOP_K     = 20   # 每筆 test row 從 BM25 額外補充最多 20 個候選 labels

# ==== Sample submission 骨架 ====
sample     = pd.read_csv(SAMPLE_CSV)
ROW_ID_COL = sample.columns[0]
PRED_COL   = sample.columns[1]
print(f"Sample shape: {sample.shape}")


# ── BM25 ──────────────────────────────────────────────────────────────────────

def tokenize_text(text: str) -> list:
    if not isinstance(text, str):
        return []
    return re.findall(r'\w+', text.lower())


class LightweightBM25:
    """純 Python 倒排索引 BM25，無外部依賴。"""

    def __init__(self, docs: list, k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b  = b
        self.doc_lens    = [len(d) for d in docs]
        self.avg_doc_len = sum(self.doc_lens) / max(len(docs), 1)
        self.N           = len(docs)

        self.index: dict = {}
        df: dict = {}
        for doc_id, doc in enumerate(docs):
            counts = Counter(doc)
            for term, tf in counts.items():
                self.index.setdefault(term, []).append((doc_id, tf))
                df[term] = df.get(term, 0) + 1

        self.idf = {
            term: math.log(1 + (self.N - f + 0.5) / (f + 0.5))
            for term, f in df.items()
        }

    def retrieve_top_k(self, query_tokens: list, k: int) -> list:
        scores: dict = {}
        for term in set(query_tokens):
            if term not in self.index:
                continue
            idf = self.idf[term]
            for doc_id, tf in self.index[term]:
                num = idf * tf * (self.k1 + 1)
                den = tf + self.k1 * (
                    1 - self.b + self.b * self.doc_lens[doc_id] / self.avg_doc_len
                )
                scores[doc_id] = scores.get(doc_id, 0.0) + num / den
        if not scores:
            return list(range(min(k, self.N)))
        return sorted(scores, key=scores.__getitem__, reverse=True)[:k]


# ── 訓練集 & BM25 索引 ────────────────────────────────────────────────────────

print("Loading train set & building BM25 index ...")
train_df = pd.read_csv(TRAIN_CSV)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    train_df[col] = train_df[col].fillna("")

train_df["target"] = (
    train_df["Category"].astype(str) + ":" +
    train_df["Misconception"].fillna("NA").astype(str)
)
unique_labels     = sorted(train_df["target"].unique().tolist())
unique_labels_set = set(unique_labels)
fallback_labels   = train_df["target"].value_counts().head(3).index.tolist()
if not fallback_labels:
    fallback_labels = ["NA:NA", "NA:NA", "NA:NA"]
print(f"# unique labels: {len(unique_labels)}, fallback: {fallback_labels}")

train_corpus = [
    tokenize_text(
        r["QuestionText"] + " " + r["MC_Answer"] + " " + r["StudentExplanation"]
    )
    for _, r in train_df.iterrows()
]
bm25         = LightweightBM25(train_corpus)
train_labels = train_df["target"].tolist()
print(f"BM25 index built over {len(train_corpus)} docs.")


def get_bm25_candidate_labels(row: pd.Series, k: int = BM25_TOP_K) -> list:
    """BM25 top-k 訓練例 labels（去重，限定在 unique_labels_set 內）。"""
    query   = tokenize_text(
        row["QuestionText"] + " " + row["MC_Answer"] + " " + row["StudentExplanation"]
    )
    top_idx = bm25.retrieve_top_k(query, k=k)
    seen, result = set(), []
    for i in top_idx:
        lbl = train_labels[i]
        if lbl in unique_labels_set and lbl not in seen:
            result.append(lbl)
            seen.add(lbl)
    return result


# ── Prompt（與 v5.1 完全相同，不加 few-shot 避免截斷）────────────────────────

def build_user_prompt(question, correct_answer, student_explanation):
    return (
        "You are a math misconception classifier.\n"
        "Given the question, the correct answer, and the student's explanation, "
        "predict the final label in the format `Category:Misconception`.\n"
        "If the category is not a misconception type, use `NA` for the misconception part.\n\n"
        f"Question: {question}\n"
        f"Correct answer: {correct_answer}\n"
        f"Student explanation: {student_explanation}\n\n"
        "Return only the label."
    )

def build_prompt(row):
    user = build_user_prompt(
        row["QuestionText"], row["MC_Answer"], row["StudentExplanation"]
    )
    return f"<start_of_turn>user\n{user}<end_of_turn>\n<start_of_turn>model\n"


# ── 模型載入（與 v5.1 相同）──────────────────────────────────────────────────

print("Loading base model in bf16 ...")
t0        = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_PATH)
tokenizer.padding_side = "left"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_PATH,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)
print(f"  base loaded in {time.time()-t0:.1f}s")

t0    = time.time()
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
print(f"  adapter loaded in {time.time()-t0:.1f}s")

print("Merging LoRA into base ...")
t0    = time.time()
model = model.merge_and_unload()
torch.cuda.empty_cache()
print(f"  merge done in {time.time()-t0:.1f}s")

model.eval()
model.config.use_cache = True
device = next(model.parameters()).device

pad_id = tokenizer.pad_token_id
if pad_id is None:
    pad_id = tokenizer.eos_token_id
    tokenizer.pad_token_id = pad_id

end_of_turn_id = tokenizer.convert_tokens_to_ids("<end_of_turn>")
stop_ids = [tokenizer.eos_token_id]
if end_of_turn_id and end_of_turn_id != tokenizer.unk_token_id:
    stop_ids.append(end_of_turn_id)


# ── Test 讀取 ─────────────────────────────────────────────────────────────────

test_df = pd.read_csv(TEST_CSV).reset_index(drop=True)
for col in ["QuestionText", "MC_Answer", "StudentExplanation"]:
    test_df[col] = test_df[col].fillna("")
print(f"Test size: {len(test_df)}")
assert len(test_df) == len(sample)


# ── Phase 1: Beam Search ──────────────────────────────────────────────────────

def clean_label(text: str) -> str:
    if not text:
        return ""
    label = text.splitlines()[0].strip()
    if " " in label:
        label = label.split(" ")[0]
    return label

@torch.no_grad()
def beam_generate_batch(prompts: list) -> list:
    tokenizer.padding_side = "left"
    enc = tokenizer(
        prompts, return_tensors="pt", padding=True,
        truncation=True, max_length=1024,
    ).to(device)

    outputs = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        num_return_sequences=NUM_RETURN,
        do_sample=False,
        early_stopping=True,
        eos_token_id=stop_ids,
        pad_token_id=tokenizer.pad_token_id,
        use_cache=True,
    )
    prompt_len = enc["input_ids"].shape[1]
    outputs    = outputs.view(len(prompts), NUM_RETURN, -1)

    results = []
    for i in range(len(prompts)):
        valid, seen = [], set()
        for k in range(NUM_RETURN):
            gen   = outputs[i, k, prompt_len:]
            text  = tokenizer.decode(gen, skip_special_tokens=True).strip()
            label = clean_label(text)
            if label in unique_labels_set and label not in seen:
                valid.append(label)
                seen.add(label)
        results.append(valid)
    return results


# ── Phase 2: Log-likelihood Re-ranking ───────────────────────────────────────

@torch.no_grad()
def score_candidates_batched(prompts: list, candidates_per_prompt: list) -> list:
    """Right-padding + SUB_BATCH_SIZE 分批 forward，logits 先移回 CPU 再計算。"""
    flat_sequences: list = []
    flat_meta: list = []  # (prompt_idx, cand_idx, label_token_len)

    for pi, (prompt, cands) in enumerate(zip(prompts, candidates_per_prompt)):
        if not cands:
            continue
        prompt_ids = tokenizer.encode(prompt, add_special_tokens=True)
        for ci, cand in enumerate(cands):
            cand_ids = tokenizer.encode(cand, add_special_tokens=False)
            cand_ids.append(end_of_turn_id if end_of_turn_id else tokenizer.eos_token_id)
            flat_sequences.append(prompt_ids + cand_ids)
            flat_meta.append((pi, ci, len(cand_ids)))

    if not flat_sequences:
        return [[] for _ in prompts]

    B       = len(flat_sequences)
    max_len = max(len(s) for s in flat_sequences)

    input_ids      = torch.full((B, max_len), pad_id, dtype=torch.long)
    attention_mask = torch.zeros((B, max_len), dtype=torch.long)
    label_starts   = []

    for j, seq in enumerate(flat_sequences):
        input_ids[j, :len(seq)]      = torch.tensor(seq, dtype=torch.long)
        attention_mask[j, :len(seq)] = 1
        label_starts.append(len(seq) - flat_meta[j][2])

    all_logits = []
    for m in range(0, B, SUB_BATCH_SIZE):
        sub_ids    = input_ids[m:m + SUB_BATCH_SIZE].to(device)
        sub_mask   = attention_mask[m:m + SUB_BATCH_SIZE].to(device)
        sub_logits = model(input_ids=sub_ids, attention_mask=sub_mask).logits
        all_logits.append(sub_logits.cpu())  # 移回 CPU 節省顯存

    logits = torch.cat(all_logits, dim=0)

    scores_per_prompt = [[0.0] * len(c) for c in candidates_per_prompt]
    for j, (pi, ci, L) in enumerate(flat_meta):
        ls         = label_starts[j]
        slice_lgt  = logits[j, ls - 1:ls - 1 + L, :].float()
        log_probs  = torch.log_softmax(slice_lgt, dim=-1)
        target_ids = flat_sequences[j][-L:]
        target     = torch.tensor(target_ids)
        tok_lp     = log_probs.gather(1, target.unsqueeze(1)).squeeze(1)
        scores_per_prompt[pi][ci] = tok_lp.mean().item()

    return scores_per_prompt


# ── 主推論迴圈 ────────────────────────────────────────────────────────────────

print("\nStart inference v5.2 (BM25 Candidate Expansion) ...")
pred_dict: dict = {}
phase1_time = bm25_time = phase2_time = 0.0
n_empty_beam = total_beam_cands = total_merged_cands = 0

try:
    for start in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Batch"):
        batch_df = test_df.iloc[start:start + BATCH_SIZE]
        prompts  = [build_prompt(r) for _, r in batch_df.iterrows()]

        # Phase 1: Beam Search
        t0             = time.time()
        beam_candidates = beam_generate_batch(prompts)
        phase1_time    += time.time() - t0

        # BM25 Candidate Expansion
        t0              = time.time()
        bm25_candidates = [get_bm25_candidate_labels(r) for _, r in batch_df.iterrows()]
        bm25_time      += time.time() - t0

        # Merge: beam 優先，BM25 補後面
        merged_candidates = []
        for beam_c, bm25_c in zip(beam_candidates, bm25_candidates):
            seen     = set(beam_c)
            combined = list(beam_c)
            for lbl in bm25_c:
                if lbl not in seen:
                    combined.append(lbl)
                    seen.add(lbl)
            merged_candidates.append(combined)

        # Phase 2: Score ALL candidates (beam + BM25)
        t0          = time.time()
        scores      = score_candidates_batched(prompts, merged_candidates)
        phase2_time += time.time() - t0

        # Top-3
        for i, (_, row) in enumerate(batch_df.iterrows()):
            beam_c   = beam_candidates[i]
            merged_c = merged_candidates[i]
            total_beam_cands   += len(beam_c)
            total_merged_cands += len(merged_c)

            if not beam_c:
                n_empty_beam += 1

            if not merged_c:
                top3 = list(fallback_labels[:3])
            else:
                ranked = sorted(zip(merged_c, scores[i]), key=lambda x: -x[1])
                top3   = [c for c, _ in ranked]

            for fb in fallback_labels:
                if len(top3) >= 3:
                    break
                if fb not in top3:
                    top3.append(fb)

            while len(top3) < 3:
                top3.append(fallback_labels[0])

            pred_dict[row["row_id"]] = " ".join(top3[:3])

except Exception as e:
    print(f"\n[CRITICAL ERROR] {e}")
    for _, row in test_df.iterrows():
        if row["row_id"] not in pred_dict:
            pred_dict[row["row_id"]] = " ".join(fallback_labels[:3])


# ── Diagnostic ────────────────────────────────────────────────────────────────

n = len(test_df)
print(f"\nTiming:")
print(f"  Phase 1 (beam):        {phase1_time:.1f}s")
print(f"  BM25 expansion:        {bm25_time:.1f}s")
print(f"  Phase 2 (re-rank):     {phase2_time:.1f}s")
print(f"  Avg beam cands/row:    {total_beam_cands / n:.2f}")
print(f"  Avg merged cands/row:  {total_merged_cands / n:.2f}")
print(f"  Rows w/ 0 beam cands:  {n_empty_beam} ({n_empty_beam / n * 100:.1f}%)")


# ── Submission ────────────────────────────────────────────────────────────────

submission          = sample.copy()
submission[PRED_COL] = submission[ROW_ID_COL].map(pred_dict)

if submission[PRED_COL].isna().any():
    submission[PRED_COL] = submission[PRED_COL].fillna(" ".join(fallback_labels[:3]))

print("\nValidation:")
print(f"  Shape    : {submission.shape}")
print(f"  Any NaN  : {submission.isna().any().any()}")
print(f"  All 3pred: {(submission[PRED_COL].str.split().str.len() == 3).all()}")

assert submission.shape == sample.shape
assert not submission.isna().any().any()
assert (submission[PRED_COL] != "").all()
assert (submission[PRED_COL].str.split().str.len() == 3).all()

submission.to_csv(OUTPUT_CSV, index=False)
print(f"\n[OK] Saved {OUTPUT_CSV}")


Sample shape: (3, 2)
Loading train set & building BM25 index ...
# unique labels: 65, fallback: ['True_Correct:NA', 'False_Neither:NA', 'True_Neither:NA']
BM25 index built over 36696 docs.
Loading base model in bf16 ...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

  base loaded in 22.4s


/usr/local/lib/python3.12/dist-packages/peft/config.py:220: UserWarning: Unexpected keyword arguments ['lora_ga_config', 'use_bdlora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


  adapter loaded in 2.5s
Merging LoRA into base ...
  merge done in 0.2s
Test size: 3

Start inference v5.2 (BM25 Candidate Expansion) ...


Batch: 100%|██████████| 1/1 [00:06<00:00,  6.83s/it]


Timing:
  Phase 1 (beam):        3.1s
  BM25 expansion:        0.4s
  Phase 2 (re-rank):     3.3s
  Avg beam cands/row:    4.00
  Avg merged cands/row:  4.67
  Rows w/ 0 beam cands:  0 (0.0%)

Validation:
  Shape    : (3, 2)
  Any NaN  : False
  All 3pred: True

[OK] Saved /kaggle/working/submission.csv
